In [1]:
%pip install --quiet -U langgraph typing_extensions langchain_openrouter dotenv exa_py

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [ ]:
from typing import TypedDict

from pydantic import BaseModel, Field


class Comorbidity(BaseModel):
    name: str = Field(
        description="Standard medical name of the patient's medical condition."
    )


class Medication(BaseModel):
    name: str = Field(
        description="Actual medication name. Use real medication names only."
    )
    dosage: str | None = Field(
        default=None,
        description="Dosage exactly as stated, e.g. '500 mg'.",
    )
    frequency: str | None = Field(
        default=None,
        description=(
            "Frequency exactly as stated. When the number of daily doses is "
            "explicitly provided, normalize it as 'x1', 'x2', or 'x3'."
        ),
    )


class LabResult(BaseModel):
    analysis: str = Field(
        description=(
            "Standard Estonian analysis name or established abbreviation, "
            "e.g. 'Hb', 'CRP', 'HbA1c', 'kreatiniin'."
        )
    )
    value: str = Field(
        description=(
            "Result value exactly as stated, preserving the decimal comma, "
            "e.g. '112' or '6,5'."
        )
    )
    unit: str | None = Field(
        default=None,
        description="Unit preserved exactly as stated.",
    )
    date: str | None = Field(
        default=None,
        description="Sample date in dd.mm.yyyy format, when stated.",
    )


class Allergy(BaseModel):
    allergen: str = Field(
        description="Substance, medication, or other allergen."
    )
    reaction: str | None = Field(
        default=None,
        description="Allergic reaction exactly as described.",
    )


class Procedure(BaseModel):
    name: str = Field(
        description=(
            "Standard Estonian clinical term, including the operated side or "
            "location when stated, e.g. 'Apendektoomia', "
            "'Parema põlveliigese endoproteesimine', or 'Koloskoopia'. "
            "Imaging does not belong here; use investigations."
        )
    )
    year: str | None = Field(
        default=None,
        description="Year of the procedure, when stated.",
    )
    result: str | None = Field(
        default=None,
        description=(
            "Clinically relevant result only when stated, "
            "e.g. 'polüübid eemaldatud'."
        ),
    )


class Investigation(BaseModel):
    name: str = Field(
        description=(
            "Full name or standard unambiguous abbreviation with anatomical "
            "region, e.g. 'Kõhuõõne ultraheli', 'Peaaju MRT', or 'Rindkere KT'."
        )
    )
    date: str | None = Field(
        default=None,
        description="Exact investigation date in dd.mm.yyyy format, when stated.",
    )
    finding: str | None = Field(
        default=None,
        description=(
            "Concise clinically relevant finding. Include relevant positive "
            "findings and explicitly stated relevant negative findings."
        ),
    )


class HomeMeasurement(BaseModel):
    measurement: str = Field(
        description=(
            "Compact standard label, e.g. 'RR', 'pulss', 'veresuhkur', "
            "'kehakaal', 'SpO₂', or 'kehatemperatuur'."
        )
    )
    value: str = Field(
        description=(
            "Representative range, when available, or a single value. "
            "Preserve units as stated, e.g. '145–155/90–95 mmHg'."
        )
    )
    context: str | None = Field(
        default=None,
        description=(
            "Timing or context only when it affects interpretation, "
            "e.g. 'hommikuti', 'rahuolekus', or 'söögijärgne'."
        ),
    )


class FamilyHistoryItem(BaseModel):
    relative: str = Field(
        description="Relationship to the patient, e.g. 'ema', 'isa', or 'õde'."
    )
    condition: str = Field(
        description=(
            "Condition or hereditary finding, e.g. '2. tüüpi diabeet', "
            "'müokardiinfarkt', or 'rinnavähk'."
        )
    )
    detail: str | None = Field(
        default=None,
        description=(
            "Clinically relevant age, timing, or other detail, e.g. "
            "'müokardiinfarkt 52-aastaselt'."
        ),
    )


class AnamnesisState(BaseModel):
    chief_complaint: str = Field(
        description=(
            "Primary reason or reasons for today's encounter. Include only "
            "complaints or concerns explicitly discussed in the transcript."
        )
    )

    history_of_present_illness: str = Field(
        description=(
            "Chronological clinical history of the current complaint or "
            "complaints, including onset, duration, course, severity, associated "
            "symptoms, modifying factors, and previous management when stated."
        )
    )

    comorbidities: list[Comorbidity] = Field(
        default_factory=list,
        description="Known active or historically relevant medical conditions.",
    )

    medications: list[Medication] = Field(
        default_factory=list,
        description="Current medications explicitly mentioned in the transcript.",
    )

    laboratory_results: list[LabResult] = Field(
        default_factory=list,
        description="Laboratory results explicitly mentioned in the transcript.",
    )

    allergies: list[Allergy] = Field(
        default_factory=list,
        description="Known allergies and reactions explicitly mentioned.",
    )

    procedures_operations: list[Procedure] = Field(
        default_factory=list,
        description="Previous procedures and operations explicitly mentioned.",
    )

    investigations: list[Investigation] = Field(
        default_factory=list,
        description="Previous or current diagnostic investigations.",
    )

    alcohol_use: str | None = Field(
        default=None,
        description=(
            "Alcohol use, including amount, frequency, and duration when stated."
        ),
    )

    tobacco_use: str | None = Field(
        default=None,
        description=(
            "Current or previous tobacco use, including amount, frequency, "
            "duration, and cessation timing when stated."
        ),
    )

    home_monitoring: list[HomeMeasurement] = Field(
        default_factory=list,
        description="Measurements taken by the patient outside the clinic.",
    )

    family_history: list[FamilyHistoryItem] = Field(
        default_factory=list,
        description="Clinically relevant family history.",
    )

    social_history: str | None = Field(
        default=None,
        description=(
            "Relevant living situation, occupation, functional status, support "
            "network, exercise, diet, or other social circumstances."
        ),
    )


class PhysicalExamState(BaseModel):
    findings: str | None = Field(
        default=None,
        description=(
            "Physical examination findings explicitly stated in the transcript. "
            "Do not infer findings that were not documented."
        ),
    )


class DecisionState(BaseModel):
    assessment: str | None = Field(
        default=None,
        description="Clinical assessment or diagnostic impression.",
    )
    plan: str | None = Field(
        default=None,
        description=(
            "Agreed management plan, including treatment, investigations, "
            "referrals, follow-up, and safety-netting."
        ),
    )


class NoteStructure(BaseModel):
    anamnesis: AnamnesisState
    physical_exam: PhysicalExamState
    decision: DecisionState


class NoteState(TypedDict, total=False):
    transcript: str
    note_structure: NoteStructure

In [34]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="openai/gpt-5.3-chat",
    temperature=0
)

analysis_model = model.with_structured_output(
    NoteStructure
)

# writing_model = model.with_structured_output(
#     WritingState
# )


In [30]:
from langchain_core.messages import HumanMessage, SystemMessage


def analyse_question(state: NoteState) -> dict:
    system_message = SystemMessage(
        content="""
You are a clinical information extraction system.

Your task is to extract clinically relevant facts from a patient-clinician
transcript. You are not writing a clinical note and you must not produce
polished narrative prose.

Rules:

1. Extract only information supported by the transcript.
2. Represent information as atomic facts.
3. Preserve uncertainty, negation, temporality and change over time.
4. Distinguish patient-reported information from clinician-observed findings.
5. Do not turn a clinician's general explanation into a patient-specific fact.
6. Do not infer a diagnosis from symptoms alone.
7. Record a diagnosis only when it is explicitly discussed by the clinician,
   and preserve whether it is confirmed, suspected, differential or less likely.
8. Do not infer left or right laterality when it is not stated.
9. Do not infer an exact calendar date from relative expressions unless the
   encounter date makes the conversion unambiguous.
10. An ordered laboratory test is not a laboratory result.
11. Extract unnamed treatments. When a medication class or generic description
    is mentioned but its name is absent, set the medication name to null and
    preserve the description.
12. Empty lists mean that the information was not mentioned. They do not mean
    that the patient definitively does not have the condition.
13. Every extracted fact should contain a short supporting excerpt and timestamp
    when available.
14. Use Estonian for normalized clinical labels. Preserve source meaning and
    do not improve or embellish the clinical content.
"""
    )

    human_message = HumanMessage(
        content=state["transcript"]
    )


    result = analysis_model.invoke(
        [system_message, human_message]
    )

    return {
        "note_structure": result
    }

#def write_answer(state: NoteState) -> dict:
    analysis = state["analysis"]

    system_message = SystemMessage(
        content="""Write a well structured essay based on the analysis provided.""")
    
    user_msg = HumanMessage(
        content=f"""
Based on the following analysis, write a well structured essay. Body can containt multiple paragraphs.
{analysis.model_dump_json(indent=2)}
""" 
    )
    result = writing_model.invoke(
        [system_message, user_msg]
    )
    return {"writing": result}

In [35]:
from langgraph.graph import StateGraph, START, END


builder = StateGraph(NoteState)

builder.add_node("analyse_question", analyse_question)
#builder.add_node("write_answer", write_answer)

builder.add_edge(START, "analyse_question")
##builder.add_edge("analyse_question", "write_answer")
builder.add_edge("analyse_question", END)

graph = builder.compile()

In [36]:
result = graph.invoke(
    {"transcript": "[00:00] Nii, aga siis selle seadeldisega lindistame. See jääb teile. Ma korra ütlen lihtsalt, et meil on korduv vastuvõtt, onju. Ja täna on antibiootikumi üheksas päev, mina vaatasin. Neljapäev alustasime jah, kahekümne esimesel siis on kaheksas päev, onju. [00:18] Täna hommikul sai kaheksa nagu täis, siis õhtul läheks üheksas. Kuidas nüüd seis on? [00:24] See valu on suuremalt osalt kadunud, aga õrnalt jääb ikka. Pigem selline pakitsus ja nagu pingetunne. Ja see punetus, noh, vahel nagu lööb tagasi, eriti õhtul. [00:38] Turse on kuidas, kas on tagasi tõmmanud või pigem sama? [00:42] Ütleme, jala pealt on natuke vähem, varbaid saab paremini liigutada. Aga hüppeliigese juures on nagu rohkem tulnud ja tundub, et läheb veidi ülespoole säärde. [00:54] Et teile tundub, et ta on nagu liikunud üles, aga samas üldine jämedus on veidi vähem? [01:00] Jah, täpselt. Ei ole enam nii “kõva” turse, aga ikka paras jäme on. [01:07] Okei. Kas vahepeal on palavikku ka olnud või külmavärinaid? [01:12] Öösel nagu oli, ma ei tea, kas päris palavik, aga temperatuur on tõusnud. Mul on see nutiseade… kell, mis salvestab öiseid näite. [01:24] Te mõõtsite siis kellaga, jah? [01:26] Jah. Oih vabandust, siin ta näitab. Eelmine nädal oli madalam ja siis paar ööd järjest tõusis, mingi null koma seitse, siis üks koma üks, siis üks koma kaks või nii. [01:40] Kas ta näitab ka, mis see baseline on, millega ta võrdleb? [01:44] Baseline’i ma ei ole sisse pannud, ta ise arvutab. Mul endal baseline on tavaliselt alla 36, mingi 35,8 umbes. [01:53] No see ongi huvitav, et öösiti kehatemperatuur tavaliselt langeb, ja kui ta pigem tõuseb, siis midagi kehas toimub. Aga päris palavikuks me üle 37,5 pigem loeme, eks. Päeval mõõtsite ka? [02:07] Päeval kraadisin, oli 36,9 ja üks kord 37,0. Nii et mitte kõrge, aga nagu kõrgem kui muidu. [02:16] Okei. Muud näitajad, pulss või hingamine, kas kell näitab ka midagi “väljas ringest”? [02:22] Jah, ta näitab, et taastumine või midagi on kehvem. Pulss on natuke kõrgem, mul tavaliselt on 54, nüüd oli öösel 60 ringis. [02:32] Infektsiooni või põletikuga võib pulss tõusta küll, kuigi numbriliselt on see ikkagi madal. Muud liigesed ei valuta praegu? [02:40] Ei, muud liigesed ei valuta. [02:43] Antibiootikumiga kõrvaltoimeid, kõhuga muret, löövet? [02:47] Ei, absoluutselt ei ole. Kõht on okei. [02:52] Eelmine kord teil läks ka päris pikalt enne kui jalg rahunes, eks? [02:56] Jah, tookord oli vist peaaegu kuu aega. Aga siis ma ei pidanud teist antibiootikumi juurde võtma. [03:03] Siis tegime selle pika kümnepäevase kuuri ja jäi vaikselt maha. Praegu ma vaataks selle jala üle ja laseks veenivereanalüüsi võtta: üldvere, põletikunäitajad, ja igaks juhuks kusihape ka. Kuigi kui on podagra atakk, siis kusihape võib olla hoopis madalam ja see analüüs pole 100% usaldusväärne. [03:24] Ma ise ka ei arva, et podagra, sest see algas pigem siit talla juurest ja siis tuli üles. Aga noh, parem kontrollida. [03:32] Just. Olge hea, võtke sokid ja jalanõud ära, mõlemalt jalalt, siis on hea võrrelda. Ja heitke pikali, mul on lihtsam vaadata. [03:43] Jah. Ma natuke kartsin, et kui ma ei saa millegagi hakkama… et talvel on keeruline, kui jalg nii turses. [03:51] No jah, talvel on see eriti tüütu. Okei, ma vaatan… värv on täna parem kui oli? [03:59] On küll, eelmine nädal oli tumepunane, peaaegu tulipunane. Mul isegi pilt on telefonis. [04:06] Turse ulatub tegelikult siia säärde ka, onju. Ja katsudes on soojem kui teine jalg, jah. Siit vajutades… kas siit on valus? [04:17] Siit ei ole. Siit külje pealt ka eriti ei ole. [04:22] Liigutades hüppeliigest, kas teeb haiget? [04:25] Ei, liigutades ei ole. Ainult pöidla all, nagu päka juures, on natuke hell. [04:32] Jah, siin on kõige rohkem turses ka. Siit alt… okei. Mujalt väga ei valuta. Hea. Võite tagasi panna. [04:43] Mulle endale tundub, et ta on isegi natuke rohkem turses kui eelmisel korral, kui ma käisin. [04:49] Eelmine kord, jah, kui te käisite [spetsialist] juures ja siis minu juures ka. Siis see alles algas, te jõudsite kiiresti arstile, see on hea. [04:59] No seda rohtu ma praegu ei tahaks kohe vahetada, kuigi võib-olla on seda vaja. Teeme enne analüüsid ära ja otsustame selle alusel. Praegu võtate edasi, kokku 12 päeva, nii et umbes kolm päeva on veel minna, lõpeb vist laupäeva õhtul või pühapäeva hommikul. [05:16] Okei. [05:18] Nii, analüüsidest: hemogramm, CRP ja muud põletikunäitajad, ja kusihape. Kui me juba torkame, siis teeme ära. Ma vaatan kohe, kuhu te saate minna. [05:30] Üks hetk… lähete kabinet neli. Ma annan õele märku ka, et te tulete. Veeniveri siis. [05:39] Hästi. [05:40] Ja homme ma helistan vastustega, kas sobib? Homseks peaks olemas olema. [05:45] Sobib küll. [05:47] Okei. Siis kabinet neli, analüüsid. Praegu võtate antibiootikumi edasi. Jalga hoiate võimalusel üleval, kui istute, siis pigem sirgelt ja natuke kõrgemal. Ja jahedat võib peale panna, mitte jääkülma, aga siukest jahedat kompressi. [06:05] Ma olen õhtuti teinud soolvee leotist, sellist sooja… see vist ei ole hea? [06:10] Soe võib turset süvendada, jah. Kui te tunnete, et leevendab, siis võib, aga pigem soovitatakse jahedat. Ja see on ka oluline, et kui jalg on üleval ja siis lasete alla, siis veri nagu vajub jalga tagasi ja siis võib olla kehvem hetk. [06:28] Arusaadav. Ma siis proovin rohkem jahedat ja hoian üleval. [06:33] Just. Olge nii, nii mugavalt kui võimalik, piinlema ei pea. Kas midagi veel küsida praegu? [06:40] Ei, praegu rohkem ei ole. Siis räägime homme. [06:44] Hästi. Selle ma panen kinni."
}
)

result

{'transcript': '[00:00] Nii, aga siis selle seadeldisega lindistame. See jääb teile. Ma korra ütlen lihtsalt, et meil on korduv vastuvõtt, onju. Ja täna on antibiootikumi üheksas päev, mina vaatasin. Neljapäev alustasime jah, kahekümne esimesel siis on kaheksas päev, onju. [00:18] Täna hommikul sai kaheksa nagu täis, siis õhtul läheks üheksas. Kuidas nüüd seis on? [00:24] See valu on suuremalt osalt kadunud, aga õrnalt jääb ikka. Pigem selline pakitsus ja nagu pingetunne. Ja see punetus, noh, vahel nagu lööb tagasi, eriti õhtul. [00:38] Turse on kuidas, kas on tagasi tõmmanud või pigem sama? [00:42] Ütleme, jala pealt on natuke vähem, varbaid saab paremini liigutada. Aga hüppeliigese juures on nagu rohkem tulnud ja tundub, et läheb veidi ülespoole säärde. [00:54] Et teile tundub, et ta on nagu liikunud üles, aga samas üldine jämedus on veidi vähem? [01:00] Jah, täpselt. Ei ole enam nii “kõva” turse, aga ikka paras jäme on. [01:07] Okei. Kas vahepeal on palavikku ka olnud või külmavärin

In [37]:
from IPython.display import Markdown, display


def show_clinical_note(note_data: dict) -> None:
    anamnesis = note_data["anamnesis"]
    physical_exam = note_data["physical_exam"]
    decision = note_data["decision"]

    def render_items(items, formatter):
        if not items:
            return "_Ei ole dokumenteeritud._"
        return "\n".join(f"- {formatter(item)}" for item in items)

    comorbidities = render_items(
        anamnesis.get("comorbidities", []),
        lambda x: x["name"],
    )

    medications = render_items(
        anamnesis.get("medications", []),
        lambda x: " ".join(
            part
            for part in [
                x.get("name"),
                x.get("dosage"),
                x.get("frequency"),
            ]
            if part
        ),
    )

    laboratory_results = render_items(
        anamnesis.get("laboratory_results", []),
        lambda x: " ".join(
            part
            for part in [
                x.get("analysis"),
                x.get("value"),
                x.get("unit"),
                x.get("date"),
            ]
            if part
        ),
    )

    home_monitoring = render_items(
        anamnesis.get("home_monitoring", []),
        lambda x: (
            f"**{x['measurement'].capitalize()}:** {x['value']}"
            + (f" — {x['context']}" if x.get("context") else "")
        ),
    )

    family_history = render_items(
        anamnesis.get("family_history", []),
        lambda x: (
            f"{x['relative']}: {x['condition']}"
            + (f" ({x['detail']})" if x.get("detail") else "")
        ),
    )

    markdown = f"""
# Kliiniline märge

## Anamnees

### Pöördumise põhjus

{anamnesis.get("chief_complaint", "_Ei ole dokumenteeritud._")}

### Käesoleva haiguse anamnees

{anamnesis.get("history_of_present_illness", "_Ei ole dokumenteeritud._")}

### Kaasuvad haigused

{comorbidities}

### Ravimid

{medications}

### Laboratoorsed analüüsid

{laboratory_results}

### Kodused mõõtmised

{home_monitoring}

### Perekonnaanamnees

{family_history}

## Füüsiline läbivaatus

{physical_exam.get("findings", "_Ei ole dokumenteeritud._")}

## Hinnang

{decision.get("assessment", "_Ei ole dokumenteeritud._")}

## Plaan

{decision.get("plan", "_Ei ole dokumenteeritud._")}
"""

    display(Markdown(markdown))

In [38]:
show_clinical_note(result["note_structure"].model_dump(exclude_none=True))


# Kliiniline märge

## Anamnees

### Pöördumise põhjus

Jalavalu, turse ja punetus antibiootikumravi ajal

### Käesoleva haiguse anamnees

Korduv visiit. Patsient on antibiootikumi kuuri 8.–9. päeval. Valu on suuremalt osalt taandunud, kuid püsib kerge pakitsus ja pingetunne. Punetus vahel ägeneb, eriti õhtuti. Turse jalalaba piirkonnas on veidi vähenenud, varvaste liikuvus paranenud, kuid hüppeliigese juures turse suurenenud ja levinud sääre suunas. Patsient kirjeldab, et turse on vähem "kõva", kuid endiselt märgatav. Öösiti esinenud kehatemperatuuri tõusu (kella andmetel +0,7 kuni +1,2 üle tavapärase), päeval mõõdetud 36,9–37,0. Pulss öösel tõusnud tavapäraselt 54-lt umbes 60-ni. Muud liigesed ei valuta. Antibiootikumravil kõrvaltoimed puuduvad.

### Kaasuvad haigused

_Ei ole dokumenteeritud._

### Ravimid

- antibiootikum

### Laboratoorsed analüüsid

_Ei ole dokumenteeritud._

### Kodused mõõtmised

- **Kehatemperatuur:** 36,9–37,0 — päeval
- **Kehatemperatuur:** +0,7 kuni +1,2 üle tavapärase — öösel nutiseadme järgi
- **Pulss:** ~60 — öösel, tavapäraselt 54

### Perekonnaanamnees

_Ei ole dokumenteeritud._

## Füüsiline läbivaatus

Jalg: turse ulatub säärele, katsudes soojem kui teine jalg. Valu puudub enamikus piirkondades, hellus pöia piirkonnas pöidla all. Hüppeliigese liigutamine ei ole valulik.

## Hinnang

Püsiv alajäseme põletikuline seisund antibiootikumravi foonil; podagra kahtlus vähetõenäoline, kuid diferentsiaaldiagnostikas.

## Plaan

Jätkata antibiootikumi kokku 12 päeva. Tellitud vereanalüüsid: hemogramm, CRP/põletikunäitajad, kusihape. Soovitus hoida jalg kõrgemal, kasutada jahedat kompressi. Järelkontakt järgmisel päeval analüüside vastustega.
